In [ ]:
# Cell 1: Install + Clone + Data\n\nimport numpy as np\nnp.int = int; np.float = float\n\n!pip install torch --index-url https://download.pytorch.org/whl/cpu -q\n!pip install scipy pandas numba tqdm numpy biopython msgpack scikit-learn PyYAML -q\n\n!rm -rf Enhanced-ECNet\n!git clone https://github.com/itozheng-max/Enhanced-ECNet.git\n%cd Enhanced-ECNet\n\n!wget -q https://raw.githubusercontent.com/itozheng-max/Enhanced-ECNet/main/ecnet_rrm_data.tar.gz\n!tar xzf ecnet_rrm_data.tar.gz --strip-components=1\n!ls -lh data/\n\nimport sys, time, torch\nsys.path.insert(0, \".\")\nprint(f\"Py {sys.version.split()[0]} | Torch {torch.__version__} | GPU {torch.cuda.is_available()}\")

In [ ]:
# Cell 2: Import

from ecnet.ecnet import ECNet
from ecnet.spatial_mask import SpatialMask

FASTA     = "data/RRM.fasta"
SINGLE_TSV = "data/RRM_single.tsv"
DOUBLE_TSV = "data/RRM_double.tsv"
BRAW      = "data/RRM.braw"
DIST_NPY  = "data/distance_matrix.npy"

print("OK")

In [ ]:
# Cell 3: Interactive UI\n\nimport ipywidgets as widgets\nfrom IPython.display import display, Markdown, clear_output\nimport pickle, os\n\nkernel_select = widgets.Dropdown(\n    options=['baseline', 'sigmoid', 'hill', 'surprise'],\n    value='hill',\n    description='Kernel:'\n)\n\ntask_select = widgets.Dropdown(\n    options=[('Single->Single', 'ss'), ('Single->Double', 'sd'), ('Double->Double', 'dd')],\n    value='sd',\n    description='Task:'\n)\n\nd0_slider = widgets.FloatSlider(value=8.0, min=4.0, max=14.0, step=0.5, description='d0 (A)')\ngamma_slider = widgets.FloatSlider(value=1.5, min=0.5, max=3.0, step=0.1, description='gamma')\nn_slider = widgets.IntSlider(value=4, min=1, max=10, step=1, description='n')\neps_slider = widgets.FloatSlider(value=0.10, min=0.01, max=0.50, step=0.01, description='epsilon')\nalpha_slider = widgets.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.5, description='alpha')\n\nrun_btn = widgets.Button(description='Train & Save', button_style='success', icon='play')\nsave_name = widgets.Text(value='my_experiment', description='Name:', placeholder='experiment name')\n\nformula_out = widgets.Output()\nlog_out = widgets.Output()\n\nRESULTS_DIR = './results'\nos.makedirs(RESULTS_DIR, exist_ok=True)\n\ndef show_formula(kernel):\n    formulas = {\n        'baseline': r'$$ eij_{new} = eij $$',\n        'sigmoid': r'$$ W(d) = \\frac{1}{1 + e^{\\gamma (d - d_0)}} \\qquad eij_{new} = \\frac{eij}{W + \\epsilon} $$',\n        'hill': r'$$ W(d) = \\frac{1}{1 + (d/d_0)^n} \\qquad eij_{new} = \\frac{eij}{W + \\epsilon} $$',\n        'surprise': r'$$ W(d) = \\frac{1}{1 + e^{\\gamma (d - d_0)}} \\qquad eij_{new} = \\frac{eij}{W + \\epsilon} \\cdot \\left[1 + \\alpha(1-W)\\tanh(|eij|_{norm})\\right] $$',\n    }\n    formula_out.clear_output(wait=True)\n    with formula_out:\n        display(Markdown(f'### Kernel: {kernel}'))\n        display(Markdown(formulas[kernel]))\n\ndef on_kernel_change(change):\n    show_formula(change['new'])\n    k = change['new']\n    d0_slider.layout.display = 'none' if k == 'baseline' else ''\n    gamma_slider.layout.display = '' if k in ('sigmoid', 'surprise') else 'none'\n    n_slider.layout.display = '' if k == 'hill' else 'none'\n    eps_slider.layout.display = 'none' if k == 'baseline' else ''\n    alpha_slider.layout.display = '' if k == 'surprise' else 'none'\n\nkernel_select.observe(on_kernel_change, 'value')\nshow_formula(kernel_select.value)\non_kernel_change({'new': kernel_select.value})\n\ndef build_mask():\n    k = kernel_select.value\n    if k == 'baseline':\n        return None\n    m = {\n        'path': DIST_NPY,\n        'mode': 'surprise' if k == 'surprise' else 'divide',\n        'd0': d0_slider.value,\n        'epsilon': eps_slider.value,\n    }\n    if k == 'sigmoid':\n        m['kernel'] = 'sigmoid'\n        m['gamma'] = gamma_slider.value\n    elif k == 'hill':\n        m['kernel'] = 'hill'\n        m['n'] = n_slider.value\n    elif k == 'surprise':\n        m['kernel'] = 'sigmoid'\n        m['gamma'] = gamma_slider.value\n        m['alpha'] = alpha_slider.value\n    return m\n\ndef run_one(name, mask):\n    ts = task_select.value\n    train_tsv = SINGLE_TSV if ts in ('ss', 'sd') else DOUBLE_TSV\n    test_tsv = SINGLE_TSV if ts == 'ss' else DOUBLE_TSV\n    ecnet = ECNet(\n        output_dir=f\"./output/{name}\",\n        train_tsv=train_tsv, test_tsv=test_tsv,\n        fasta=FASTA, ccmpred_output=BRAW,\n        use_loc_feat=True, use_glob_feat=False,\n        split_ratio=[0.9, 0.1] if test_tsv else [0.7, 0.1, 0.2],\n        n_ensembles=1, d_embed=20, d_model=64, d_h=64,\n        batch_size=64, save_log=False,\n        spatial_mask=mask,\n    )\n    ecnet.train(epochs=200, patience=100, eval_freq=20, log_freq=200)\n    r = ecnet.test(mode=\"ensemble\", save_prediction=True)\n    return r[\"metric\"][\"corr\"], r[\"df\"]\n\ndef on_run_clicked(b):\n    run_btn.disabled = True\n    log_out.clear_output(wait=True)\n    with log_out:\n        k = kernel_select.value\n        ts = task_select.value\n        task_names = {'ss': 'Single->Single', 'sd': 'Single->Double', 'dd': 'Double->Double'}\n        print(f'Task: {task_names[ts]} | Kernel: {k}')\n        if k != 'baseline':\n            print(f'  d0={d0_slider.value} eps={eps_slider.value}', end='')\n            if k in ('sigmoid', 'surprise'): print(f' gamma={gamma_slider.value}', end='')\n            if k == 'hill': print(f' n={n_slider.value}', end='')\n            if k == 'surprise': print(f' alpha={alpha_slider.value}', end='')\n            print()\n        print()\n\n        t0 = time.time()\n        b_corr, b_df = run_one('Baseline', None)\n        print(f'  Baseline -> {b_corr:.6f}  ({time.time()-t0:.0f}s)')\n\n        mask = build_mask()\n        m_corr = None\n        if mask is not None:\n            t0 = time.time()\n            m_corr, m_df = run_one(k, mask)\n            print(f'  {k:10s} -> {m_corr:.6f}  ({time.time()-t0:.0f}s)')\n            if m_corr is not None:\n                print(f'\\n  Delta: {m_corr-b_corr:+.6f} ({(m_corr/b_corr-1)*100:+.1f}%)')\n\n        # Save\n        name = save_name.value or 'experiment'\n        ts_key = task_select.value\n        timestamp = time.strftime('%Y%m%d_%H%M%S')\n        fname = f'{RESULTS_DIR}/{name}_{ts_key}_{timestamp}.pkl'\n        data = {\n            'task': ts_key,\n            'kernel': k,\n            'params': {\n                'd0': d0_slider.value if k != 'baseline' else None,\n                'epsilon': eps_slider.value if k != 'baseline' else None,\n                'gamma': gamma_slider.value if k in ('sigmoid','surprise') else None,\n                'n': n_slider.value if k == 'hill' else None,\n                'alpha': alpha_slider.value if k == 'surprise' else None,\n            },\n            'baseline_corr': b_corr,\n            'mask_corr': m_corr,\n            'delta': (m_corr - b_corr) if m_corr is not None else None,\n            'baseline_pred_df': b_df,\n            'mask_pred_df': m_df if m_corr is not None else None,\n        }\n        with open(fname, 'wb') as f:\n            pickle.dump(data, f)\n        print(f'\\n  Saved -> {fname}')\n        print(f'  !cp {fname} /content/drive/MyDrive/  # to persist on Drive')\n\n    run_btn.disabled = False\n\nrun_btn.on_click(on_run_clicked)\n\ndisplay(Markdown('---'))\ndisplay(Markdown('### Task'))\ndisplay(task_select)\ndisplay(Markdown('---'))\ndisplay(Markdown('### Kernel & Parameters'))\ndisplay(kernel_select)\ndisplay(formula_out)\ndisplay(d0_slider)\ndisplay(gamma_slider)\ndisplay(n_slider)\ndisplay(eps_slider)\ndisplay(alpha_slider)\ndisplay(Markdown('---'))\ndisplay(Markdown('### Save'))\ndisplay(save_name)\ndisplay(run_btn)\ndisplay(Markdown('---'))\ndisplay(log_out)